In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path
import os
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from scipy.signal import savgol_filter
from scipy.stats import linregress

def find_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "data").exists() and (p / "scripts").exists():
            return p
        p = p.parent
    raise RuntimeError("Root folder not found")


ROOT = find_root()

df = pd.read_csv(ROOT / "data" / "processed" / "features" / "phase_features_long.csv")

# scegli segnale
df_sig = df[df["signal"] == "t_dorsal"].copy()

# modello
model = smf.mixedlm(
    "delta_mean_from_baseline ~ condition * phase",
    data=df_sig,
    groups=df_sig["subject_id"]
)

result = model.fit()
print(result.summary())

                        Mixed Linear Model Regression Results
Model:                 MixedLM      Dependent Variable:      delta_mean_from_baseline
No. Observations:      36           Method:                  REML                    
No. Groups:            3            Scale:                   0.1949                  
Min. group size:       6            Log-Likelihood:          -19.3039                
Max. group size:       18           Converged:               Yes                     
Mean group size:       12.0                                                          
-------------------------------------------------------------------------------------
                                           Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------------------
Intercept                                   0.003    0.423  0.008 0.994 -0.827  0.833
condition[T.wind_high]                      0.158    0.441  0.359 0.720 -0.707

In [2]:
# this part is to evaluate the model fit and the indices, but it's not fully implemented yet

DATA_PATH = ROOT / "data" / "processed" / "features" / "analysis_dataset_with_tsk.csv"
OUTPUT_DIR = ROOT / "reports" / "mechanistic_plots" / "convective_indices_example"

SUBJECT_ID = None

PHASE = "walk_mod_1"

CONDITIONS = ["no_wind", "wind_high"]

TIME_COL = "elapsed_seconds"
SUBJECT_COL = "subject_id"
CONDITION_COL = "condition"
PHASE_COL = "phase"

T_DORSAL_COL = "t_dorsal"
T_MICRO_COL = "t_microclimateback"

# Savitzky-Golay smoothing settings
WINDOW_LENGTH = 11
POLYORDER = 2

# Avoid unstable ACI when the thermal gradient is too small
DELTAT_MIN_THRESHOLD = 0.01

df = pd.read_csv(DATA_PATH)

def ensure_output_dir(path):
    os.makedirs(path, exist_ok=True)


def choose_subject(df, subject_col, subject_id=None):
    if subject_id is not None:
        return subject_id

    counts = (
        df[df[PHASE_COL] == PHASE]
        .groupby([subject_col, CONDITION_COL])
        .size()
        .reset_index(name="n_rows")
    )

    valid_subjects = []

    for subj in counts[subject_col].unique():
        sub_counts = counts[counts[subject_col] == subj]

        ok = True
        for cond in CONDITIONS:
            n = sub_counts.loc[sub_counts[CONDITION_COL] == cond, "n_rows"]
            if len(n) == 0 or n.iloc[0] < 3:
                ok = False

        if ok:
            valid_subjects.append(subj)

    if len(valid_subjects) == 0:
        raise ValueError(
            "No subject has at least 3 rows for all selected conditions "
            f"in phase {PHASE}."
        )

    return sorted(valid_subjects)[0]


def check_required_columns(df, required_cols):
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(
            f"Missing columns in dataset: {missing}\n"
            f"Available columns are:\n{list(df.columns)}"
        )


def prepare_condition_data(df, subject_id, condition, phase):
    data = df[
        (df[SUBJECT_COL] == subject_id) &
        (df[CONDITION_COL] == condition) &
        (df[PHASE_COL] == phase)
    ].copy()

    if data.empty:
        raise ValueError(
            f"No data found for subject={subject_id}, "
            f"condition={condition}, phase={phase}"
        )

    data = data.sort_values(TIME_COL).reset_index(drop=True)

    cols = [
        TIME_COL,
        SUBJECT_COL,
        CONDITION_COL,
        PHASE_COL,
        T_DORSAL_COL,
        T_MICRO_COL
    ]

    data = data[cols].copy()

    # Convert signals to numeric
    data[TIME_COL] = pd.to_numeric(data[TIME_COL], errors="coerce")
    data[T_DORSAL_COL] = pd.to_numeric(data[T_DORSAL_COL], errors="coerce")
    data[T_MICRO_COL] = pd.to_numeric(data[T_MICRO_COL], errors="coerce")

    data = data.dropna(subset=[TIME_COL])
    data = data.sort_values(TIME_COL)

    # Average duplicated timestamps, if present
    data_interp = (
        data.groupby(TIME_COL, as_index=False)
        .agg({
            SUBJECT_COL: "first",
            CONDITION_COL: "first",
            PHASE_COL: "first",
            T_DORSAL_COL: "mean",
            T_MICRO_COL: "mean"
        })
    )

    # Interpolate the two signals independently over time
    data_interp[T_DORSAL_COL] = data_interp[T_DORSAL_COL].interpolate(
        method="linear",
        limit_direction="both"
    )

    data_interp[T_MICRO_COL] = data_interp[T_MICRO_COL].interpolate(
        method="linear",
        limit_direction="both"
    )

    # Now remove only rows where interpolation still failed
    data_interp = data_interp.dropna(subset=[T_DORSAL_COL, T_MICRO_COL])

    if len(data_interp) < 3:
        raise ValueError(
            f"Not enough aligned/interpolated points for subject={subject_id}, "
            f"condition={condition}, phase={phase}. "
            f"Rows after interpolation: {len(data_interp)}"
        )

    return data_interp


def smooth_signal(y, window_length=11, polyorder=2):
    y = np.asarray(y, dtype=float)

    if len(y) < 5:
        return y


    window = min(window_length, len(y))

    if window % 2 == 0:
        window -= 1

    if window <= polyorder:
        return y

    return savgol_filter(y, window_length=window, polyorder=polyorder)


def compute_convective_indices(data):
    data = data.copy()
    data = data.sort_values(TIME_COL).reset_index(drop=True)

    condition_name = data[CONDITION_COL].iloc[0] if len(data) > 0 else "unknown"

    data = data.drop_duplicates(subset=TIME_COL)

    if len(data) < 3:
        raise ValueError(
            f"Not enough valid data points for condition={condition_name}. "
            f"Only {len(data)} rows available."
        )

    t_sec = data[TIME_COL].values.astype(float)
    t_dorsal = data[T_DORSAL_COL].values.astype(float)
    t_micro = data[T_MICRO_COL].values.astype(float)

    t_dorsal_smooth = smooth_signal(
        t_dorsal,
        window_length=WINDOW_LENGTH,
        polyorder=POLYORDER
    )

    deltaT_loc = t_dorsal_smooth - t_micro

    dTdt = np.gradient(t_dorsal_smooth, t_sec)

    CR = -dTdt

    ACI = np.full_like(CR, np.nan, dtype=float)

    valid = np.abs(deltaT_loc) >= DELTAT_MIN_THRESHOLD
    ACI[valid] = CR[valid] / deltaT_loc[valid]

    data["t_dorsal_smooth"] = t_dorsal_smooth
    data["deltaT_loc"] = deltaT_loc
    data["dTdt_C_per_s"] = dTdt
    data["CR_C_per_s"] = CR
    data["ACI_1_per_s"] = ACI
    data["valid_ACI"] = valid

    print(condition_name, "- aligned rows:", len(data))

    return data

def fit_cr_vs_deltaT(data):
    valid = (
        data["valid_ACI"] &
        np.isfinite(data["CR_C_per_s"]) &
        np.isfinite(data["deltaT_loc"])
    )

    x = data.loc[valid, "deltaT_loc"].values
    y = data.loc[valid, "CR_C_per_s"].values

    if len(x) < 3:
        return None

    result = linregress(x, y)

    return {
        "k_c_slope_1_per_s": result.slope,
        "intercept_C_per_s": result.intercept,
        "r_value": result.rvalue,
        "r_squared": result.rvalue ** 2,
        "p_value": result.pvalue,
        "std_err": result.stderr,
        "n_points": len(x)
    }
    
def fit_phase_transition(data, transition_window_s=180):
    data = data.copy()
    data = data.sort_values(TIME_COL).reset_index(drop=True)

    # Use only the first part of the phase
    t0_abs = data[TIME_COL].min()
    data = data[data[TIME_COL] <= t0_abs + transition_window_s].copy()

    if len(data) < 5:
        return None

    # Local time from phase start
    t = data[TIME_COL].values.astype(float)
    t = t - t.min()

    T = data[T_DORSAL_COL].values.astype(float)

    # Smooth before fitting
    T_smooth = smooth_signal(
        T,
        window_length=WINDOW_LENGTH,
        polyorder=POLYORDER
    )

    # Initial guesses
    T0_guess = T_smooth[0]
    T_eq_guess = T_smooth[-1]
    k_guess = 1 / 120  # about 2 min time constant

    try:
        popt, pcov = curve_fit(
            exponential_relaxation,
            t,
            T_smooth,
            p0=[T_eq_guess, T0_guess, k_guess],
            bounds=(
                [20, 20, 0],
                [45, 45, 1]
            ),
            maxfev=10000
        )

        T_eq, T0, k = popt

        T_pred = exponential_relaxation(t, T_eq, T0, k)

        ss_res = np.sum((T_smooth - T_pred) ** 2)
        ss_tot = np.sum((T_smooth - np.mean(T_smooth)) ** 2)

        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        return {
            "T_eq_transition": T_eq,
            "T0_transition": T0,
            "k_transition_1_per_s": k,
            "tau_transition_s": 1 / k if k > 0 else np.nan,
            "r_squared_transition": r_squared,
            "n_points_transition": len(data)
        }

    except Exception as e:
        return {
            "T_eq_transition": np.nan,
            "T0_transition": np.nan,
            "k_transition_1_per_s": np.nan,
            "tau_transition_s": np.nan,
            "r_squared_transition": np.nan,
            "n_points_transition": len(data),
            "transition_fit_error": str(e)
        }
        
def summarize_one_group(data, subject_id, condition, phase):
    data = compute_convective_indices(data)

    # Intra-phase model:
    # CR = k_c * deltaT_loc + b
    fit = fit_cr_vs_deltaT(data)

    # Phase-transition model:
    # T(t) = T_eq + (T0 - T_eq) exp(-k t)
    transition_fit = fit_phase_transition(data)

    valid = data["valid_ACI"] & data["ACI_1_per_s"].notna()

    summary = {
        "subject_id": subject_id,
        "condition": condition,
        "phase": phase,

        # Descriptive counts
        "n_points": len(data),
        "n_valid_ACI": valid.sum(),

        # Derived physical quantities
        "mean_deltaT_loc": data["deltaT_loc"].mean(),
        "median_deltaT_loc": data["deltaT_loc"].median(),

        "mean_CR_C_per_s": data["CR_C_per_s"].mean(),
        "median_CR_C_per_s": data["CR_C_per_s"].median(),

        "mean_ACI_1_per_s": data.loc[valid, "ACI_1_per_s"].mean(),
        "median_ACI_1_per_s": data.loc[valid, "ACI_1_per_s"].median(),
        "std_ACI_1_per_s": data.loc[valid, "ACI_1_per_s"].std(),
    }

    # Add intra-phase fit
    if fit is not None:
        summary.update(fit)
    else:
        summary.update({
            "k_c_slope_1_per_s": np.nan,
            "intercept_C_per_s": np.nan,
            "r_value": np.nan,
            "r_squared": np.nan,
            "p_value": np.nan,
            "std_err": np.nan,
            "n_points_fit": 0
        })

    # Add transition fit
    if transition_fit is not None:
        summary.update(transition_fit)
    else:
        summary.update({
            "T_eq_transition": np.nan,
            "T0_transition": np.nan,
            "k_transition_1_per_s": np.nan,
            "tau_transition_s": np.nan,
            "r_squared_transition": np.nan,
            "n_points_transition": 0
        })

    return summary, data

def run_all_subjects():
    ensure_output_dir(OUTPUT_DIR)

    df = pd.read_csv(DATA_PATH)
    df.columns = df.columns.str.strip()

    required_cols = [
        TIME_COL,
        SUBJECT_COL,
        CONDITION_COL,
        PHASE_COL,
        T_DORSAL_COL,
        T_MICRO_COL
    ]

    check_required_columns(df, required_cols)

    summaries = []
    all_time_series = []

    subjects = sorted(df[SUBJECT_COL].dropna().unique())
    conditions = sorted(df[CONDITION_COL].dropna().unique())
    phases = sorted(df[PHASE_COL].dropna().unique())

    print("Subjects:", subjects)
    print("Conditions:", conditions)
    print("Phases:", phases)

    for subject_id in subjects:
        for condition in conditions:
            for phase in phases:
                subset = df[
                    (df[SUBJECT_COL] == subject_id) &
                    (df[CONDITION_COL] == condition) &
                    (df[PHASE_COL] == phase)
                ].copy()

                if subset.empty:
                    continue

                try:
                    data = prepare_condition_data(
                        df=df,
                        subject_id=subject_id,
                        condition=condition,
                        phase=phase
                    )

                    summary, data_indices = summarize_one_group(
                        data=data,
                        subject_id=subject_id,
                        condition=condition,
                        phase=phase
                    )

                    summaries.append(summary)
                    all_time_series.append(data_indices)

                except Exception as e:
                    print(
                        f"Skipping subject={subject_id}, "
                        f"condition={condition}, phase={phase}. "
                        f"Reason: {e}"
                    )

    summary_df = pd.DataFrame(summaries)

    time_series_df = pd.concat(all_time_series, ignore_index=True)

    summary_path = os.path.join(
        OUTPUT_DIR,
        "convective_indices_subject_phase_summary.csv"
    )

    time_series_path = os.path.join(
        OUTPUT_DIR,
        "convective_indices_all_timeseries.csv"
    )

    summary_df.to_csv(summary_path, index=False)
    time_series_df.to_csv(time_series_path, index=False)

    print("\nSaved:")
    print(summary_path)
    print(time_series_path)

    return summary_df, time_series_df

def plot_temperature_and_gradient(all_data, output_path):
    plt.figure(figsize=(10, 6))

    for condition, data in all_data.items():
        plt.plot(
            data[TIME_COL],
            data["t_dorsal_smooth"],
            label=f"{condition} - T dorsal"
        )
        plt.plot(
            data[TIME_COL],
            data[T_MICRO_COL],
            linestyle="--",
            label=f"{condition} - T microclimate"
        )

    plt.xlabel("Elapsed time [min]")
    plt.ylabel("Temperature [°C]")
    plt.title("Dorsal skin and microclimate temperature")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def plot_deltaT(all_data, output_path):
    plt.figure(figsize=(10, 5))

    for condition, data in all_data.items():
        plt.plot(
            data[TIME_COL],
            data["deltaT_loc"],
            label=condition
        )

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel("Elapsed time [s]")
    plt.ylabel(r"$\Delta T_{loc}$ [°C]")
    plt.title("Local thermal driving force")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def plot_cr_vs_deltaT(all_data, fit_results, output_path):
    plt.figure(figsize=(7, 6))

    for condition, data in all_data.items():
        valid = data["valid_ACI"]

        x = data.loc[valid, "deltaT_loc"]
        y = data.loc[valid, "CR_C_per_s"]

        plt.scatter(x, y, label=condition, alpha=0.7)

        fit = fit_results.get(condition)

        if fit is not None:
            x_fit = np.linspace(x.min(), x.max(), 100)
            y_fit = (
                fit["k_c_slope_1_per_s"] * x_fit +
                fit["intercept_C_per_s"]
            )
            plt.plot(x_fit, y_fit)

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel(r"$\Delta T_{loc}$ [°C]")
    plt.ylabel("Cooling Rate [°C/s]")
    plt.title(r"Cooling Rate vs local thermal driving force")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def plot_aci_time_series(all_data, output_path):
    plt.figure(figsize=(10, 5))

    for condition, data in all_data.items():
        plt.plot(
            data[TIME_COL],
            data["ACI_1_per_s"],
            label=condition
        )

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel("Elapsed time [min]")
    plt.ylabel("ACI [1/s]")
    plt.title("Apparent Convective Index over time")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()



In [3]:
# ============================================================
# MAIN SCRIPT
# ============================================================

def main():
    ensure_output_dir(OUTPUT_DIR)

    df = pd.read_csv(DATA_PATH)

    required_cols = [
        TIME_COL,
        SUBJECT_COL,
        CONDITION_COL,
        PHASE_COL,
        T_DORSAL_COL,
        T_MICRO_COL
    ]

    check_required_columns(df, required_cols)

    subject_id = choose_subject(df, SUBJECT_COL, SUBJECT_ID)

    print("\nConvective indices example")
    print("\nRows available for selected phase:")
    print(
        df[df[PHASE_COL] == PHASE]
        .groupby([SUBJECT_COL, CONDITION_COL, PHASE_COL])
        .size()
        .reset_index(name="n_rows")
        .sort_values("n_rows", ascending=False)
    )
    print("--------------------------")
    print(f"Subject: {subject_id}")
    print(f"Phase: {PHASE}")
    print(f"Conditions: {CONDITIONS}")

    all_data = {}
    fit_results = {}

    for condition in CONDITIONS:
        data = prepare_condition_data(
            df=df,
            subject_id=subject_id,
            condition=condition,
            phase=PHASE
        )

        data = compute_convective_indices(data)

        all_data[condition] = data
        fit_results[condition] = fit_cr_vs_deltaT(data)

        output_csv = os.path.join(
            OUTPUT_DIR,
            f"convective_indices_{subject_id}_{condition}_{PHASE}.csv"
        )
        data.to_csv(output_csv, index=False)

    # Save plots
    plot_temperature_and_gradient(
        all_data,
        os.path.join(
            OUTPUT_DIR,
            f"temperature_microclimate_{subject_id}_{PHASE}.png"
        )
    )

    plot_deltaT(
        all_data,
        os.path.join(
            OUTPUT_DIR,
            f"deltaT_loc_{subject_id}_{PHASE}.png"
        )
    )

    plot_cr_vs_deltaT(
        all_data,
        fit_results,
        os.path.join(
            OUTPUT_DIR,
            f"CR_vs_deltaT_{subject_id}_{PHASE}.png"
        )
    )

    plot_aci_time_series(
        all_data,
        os.path.join(
            OUTPUT_DIR,
            f"ACI_timeseries_{subject_id}_{PHASE}.png"
        )
    )

    # Save fit summary
    fit_summary = []

    for condition, result in fit_results.items():
        if result is None:
            fit_summary.append({
                "subject": subject_id,
                "phase": PHASE,
                "k_c_slope_1_per_s": np.nan,
                "intercept_C_per_s": np.nan,
                "r_squared": np.nan,
                "p_value": np.nan,
                "n_points": 0
            })
        else:
            row = {
                "subject": subject_id,
                "phase": PHASE,
                "condition": condition
            }
            row.update(result)
            fit_summary.append(row)

    fit_summary_df = pd.DataFrame(fit_summary)

    fit_summary_path = os.path.join(
        OUTPUT_DIR,
        f"fit_summary_{subject_id}_{PHASE}.csv"
    )

    fit_summary_df.to_csv(fit_summary_path, index=False)

    print("\nFit summary:")
    print(fit_summary_df)

    print("\nFiles saved in:")
    print(OUTPUT_DIR)


#if __name__ == "__main__":
#    main()

In [4]:
def exponential_relaxation(t, T_eq, T0, k):
    return T_eq + (T0 - T_eq) * np.exp(-k * t)

def run_lmm_on_convective_indices(summary_df):
    import statsmodels.formula.api as smf

    outcomes = [
    "mean_ACI_1_per_s",
    "k_c_slope_1_per_s",
    "k_transition_1_per_s",
    "tau_transition_s",
    "T_eq_transition",
    "mean_CR_C_per_s",
    "mean_deltaT_loc"
    ]

    results = {}

    for outcome in outcomes:
        data = summary_df.dropna(subset=[outcome]).copy()

        if data.empty:
            print(f"\nSkipping {outcome}: no valid data.")
            continue

        print("\n" + "=" * 80)
        print(f"LMM for {outcome}")
        print("=" * 80)

        try:
            model = smf.mixedlm(
                f"{outcome} ~ condition * phase",
                data=data,
                groups=data["subject_id"]
            )
            result = model.fit(method="lbfgs")
            print(result.summary())
            results[outcome] = result

        except Exception as e:
            print(f"Full model failed for {outcome}: {e}")

            try:
                print(f"\nTrying reduced model for {outcome}...")
                model = smf.mixedlm(
                    f"{outcome} ~ condition + phase",
                    data=data,
                    groups=data["subject_id"]
                )
                result = model.fit(method="lbfgs")
                print(result.summary())
                results[outcome] = result

            except Exception as e2:
                print(f"Reduced model also failed for {outcome}: {e2}")

    return results

def plot_boxplot_by_condition(summary_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    variables = {
    "mean_ACI_1_per_s": "Mean ACI [1/s]",
    "k_c_slope_1_per_s": r"$k_c$ slope [1/s]",
    "k_transition_1_per_s": r"Transition $k$ [1/s]",
    "tau_transition_s": r"Transition time constant $\tau$ [s]",
    "T_eq_transition": r"Estimated $T_{eq}$ [°C]",
    "mean_CR_C_per_s": "Mean cooling rate [°C/s]",
    "mean_deltaT_loc": r"Mean $\Delta T_{loc}$ [°C]"
    }

    for var, ylabel in variables.items():
        if var not in summary_df.columns:
            print(f"Skipping {var}: column not found")
            continue

        plot_data = summary_df.dropna(subset=[var]).copy()

        if plot_data.empty:
            print(f"Skipping {var}: no valid data")
            continue

        conditions = sorted(plot_data["condition"].unique())
        values = [
            plot_data.loc[plot_data["condition"] == cond, var].values
            for cond in conditions
        ]

        plt.figure(figsize=(7, 5))
        plt.boxplot(values, labels=conditions, showmeans=True)

        plt.xlabel("Condition")
        plt.ylabel(ylabel)
        plt.title(f"{ylabel} by condition")
        plt.tight_layout()

        filename = f"boxplot_{var}_by_condition.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=300)
        plt.close()


def plot_boxplot_by_condition_phase(summary_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    variables = {
        "mean_ACI_1_per_s": "Mean ACI [1/s]",
        "k_c_slope_1_per_s": r"$k_c$ slope [1/s]",
        "mean_CR_C_per_s": "Mean cooling rate [°C/s]",
        "mean_deltaT_loc": r"Mean $\Delta T_{loc}$ [°C]"
    }

    for var, ylabel in variables.items():
        if var not in summary_df.columns:
            print(f"Skipping {var}: column not found")
            continue

        plot_data = summary_df.dropna(subset=[var]).copy()

        if plot_data.empty:
            print(f"Skipping {var}: no valid data")
            continue

        phases = sorted(plot_data["phase"].unique())
        conditions = sorted(plot_data["condition"].unique())

        x_positions = []
        values = []
        labels = []

        pos = 1
        for phase in phases:
            for condition in conditions:
                subset = plot_data[
                    (plot_data["phase"] == phase) &
                    (plot_data["condition"] == condition)
                ][var].values

                if len(subset) > 0:
                    values.append(subset)
                    x_positions.append(pos)
                    labels.append(f"{phase}\n{condition}")
                    pos += 1

            pos += 1

        plt.figure(figsize=(14, 6))
        plt.boxplot(values, positions=x_positions, showmeans=True)

        plt.xticks(x_positions, labels, rotation=90)
        plt.xlabel("Phase and condition")
        plt.ylabel(ylabel)
        plt.title(f"{ylabel} by phase and condition")
        plt.tight_layout()

        filename = f"boxplot_{var}_by_condition_phase.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=300)
        plt.close()


def plot_mean_bar_by_condition(summary_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    variables = {
        "mean_ACI_1_per_s": "Mean ACI [1/s]",
        "k_c_slope_1_per_s": r"$k_c$ slope [1/s]",
        "mean_CR_C_per_s": "Mean cooling rate [°C/s]",
        "mean_deltaT_loc": r"Mean $\Delta T_{loc}$ [°C]"
    }

    for var, ylabel in variables.items():
        if var not in summary_df.columns:
            continue

        stats = (
            summary_df
            .dropna(subset=[var])
            .groupby("condition")[var]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        if stats.empty:
            continue

        plt.figure(figsize=(7, 5))
        plt.bar(
            stats["condition"],
            stats["mean"],
            yerr=stats["std"],
            capsize=5
        )

        plt.xlabel("Condition")
        plt.ylabel(ylabel)
        plt.title(f"Mean {ylabel} by condition")
        plt.tight_layout()

        filename = f"bar_mean_{var}_by_condition.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=300)
        plt.close()


def plot_time_series_mean_by_condition(time_series_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    variables = {
        "deltaT_loc": r"$\Delta T_{loc}$ [°C]",
        "CR_C_per_s": "Cooling rate [°C/s]",
        "ACI_1_per_s": "ACI [1/s]"
    }

    for var, ylabel in variables.items():
        if var not in time_series_df.columns:
            continue

        plot_data = time_series_df.dropna(subset=[var]).copy()

        if plot_data.empty:
            continue

        # Use elapsed minutes for nicer plots
        plot_data["elapsed_minutes_plot"] = plot_data["elapsed_seconds"] / 60

        # Bin time to avoid chaotic overplotting
        plot_data["time_bin_min"] = plot_data["elapsed_minutes_plot"].round(1)

        mean_ts = (
            plot_data
            .groupby(["condition", "time_bin_min"])[var]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        plt.figure(figsize=(10, 5))

        for condition in sorted(mean_ts["condition"].unique()):
            subset = mean_ts[mean_ts["condition"] == condition]

            plt.plot(
                subset["time_bin_min"],
                subset["mean"],
                label=condition
            )

        plt.xlabel("Elapsed time [min]")
        plt.ylabel(ylabel)
        plt.title(f"Mean time series of {ylabel} by condition")
        plt.legend()
        plt.tight_layout()

        filename = f"timeseries_mean_{var}_by_condition.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=300)
        plt.close()

def run_all_convective_plots(summary_df, time_series_df, output_dir):
    plot_boxplot_by_condition(summary_df, output_dir)
    plot_boxplot_by_condition_phase(summary_df, output_dir)
    plot_mean_bar_by_condition(summary_df, output_dir)
    plot_time_series_mean_by_condition(time_series_df, output_dir)

    print("Plots saved in:")
    print(output_dir)


In [5]:
summary_df, time_series_df = run_all_subjects()
lmm_results = run_lmm_on_convective_indices(summary_df)

PLOT_DIR = OUTPUT_DIR / "summary_plots"

run_all_convective_plots(
    summary_df=summary_df,
    time_series_df=time_series_df,
    output_dir=PLOT_DIR
)


Subjects: ['tester_01', 'tester_02', 'tester_03']
Conditions: ['no_wind', 'wind_high', 'wind_low']
Phases: ['standing_1', 'standing_2', 'walk_low_1', 'walk_low_2', 'walk_mod_1', 'walk_mod_2']
no_wind - aligned rows: 300
no_wind - aligned rows: 300
no_wind - aligned rows: 300
no_wind - aligned rows: 300
no_wind - aligned rows: 300
no_wind - aligned rows: 300
wind_high - aligned rows: 360
wind_high - aligned rows: 360
wind_high - aligned rows: 360
wind_high - aligned rows: 360
wind_high - aligned rows: 360
wind_high - aligned rows: 360
wind_low - aligned rows: 360
wind_low - aligned rows: 360
wind_low - aligned rows: 360
wind_low - aligned rows: 360
wind_low - aligned rows: 360
wind_low - aligned rows: 360
no_wind - aligned rows: 360
no_wind - aligned rows: 360
no_wind - aligned rows: 360
no_wind - aligned rows: 360
no_wind - aligned rows: 360
no_wind - aligned rows: 360
wind_high - aligned rows: 360
wind_high - aligned rows: 360
wind_high - aligned rows: 360
wind_high - aligned rows: 36

/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/eleonorab/.local/lib/python3.10

              Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: k_c_slope_1_per_s
No. Observations:   36      Method:             REML             
No. Groups:         3       Scale:              0.0000           
Min. group size:    6       Log-Likelihood:     97.1576          
Max. group size:    18      Converged:          Yes              
Mean group size:    12.0                                         
-----------------------------------------------------------------
                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept               0.004    0.002  1.825 0.068 -0.000  0.009
condition[T.wind_high] -0.001    0.002 -0.377 0.706 -0.006  0.004
condition[T.wind_low]  -0.002    0.002 -0.893 0.372 -0.006  0.002
phase[T.standing_2]    -0.000    0.003 -0.043 0.965 -0.007  0.006
phase[T.walk_low_1]    -0.003    0.003 -0.839 0.402 -0.009  0.004
phase[T.walk_low_2]    -

/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/eleonorab/.local/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random ef

Plots saved in:
/home/eleonorab/repository/convective_cooling/reports/mechanistic_plots/convective_indices_example/summary_plots
